# Idea 1 analysis: wider scale coverage against height metadata +20 %
Dumps from Idea1_Scale_Coverage_Dump: 60 official photos (Test_Iteration_9's list), 15 queries per photo (headings -5/0/+5 x scales 0.80/0.85/0.90/1.00/1.05), clean and with rel_height x 1.2.
Rules compared offline: STD6 = today's six standard queries (bar 8 or agreement 2, no escalation); ALL15 = all 15 pooled, answer at bar B; STD6 then 15 = the standard rule that escalates to all 15 at bar 11 (today's rule with a wider escalation).
Fable's falsifier: the idea dies if the height +20 % score does not come within about 3 points of clean, or it costs more than 0.3 on clean data. Reference (Test_Iteration_9, 27 queries, scales 0.95-1.05): clean 48.8, height +20 % 31.8.


In [ ]:
import os, sys
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
from train.checks import replay
T7 = ROOT / "tests/Test_Iteration_7/results"
STD = [(h, s) for s in (1.0, 1.05) for h in (-5.0, 0.0, 5.0)]
ALL = [(h, s) for s in (0.80, 0.85, 0.90, 1.0, 1.05) for h in (-5.0, 0.0, 5.0)]
def decide(photo, rule, bar=8):
    ids_std, ids_all = replay.query_ids(photo, STD), replay.query_ids(photo, ALL)
    if rule == "ALL15":
        cand = replay.pooled(photo, ids_all); k = replay.choose(cand); n = 15
        x, y, inl = (cand["x"][k], cand["y"][k], cand["inliers"][k]) if k >= 0 else (0, 0, 0)
        ans = k >= 0 and inl >= bar
    else:
        cand = replay.pooled(photo, ids_std); k = replay.choose(cand); n = 6
        x, y, inl = (cand["x"][k], cand["y"][k], cand["inliers"][k]) if k >= 0 else (0, 0, 0)
        agree = replay.agreement(photo, ids_std, x, y, photo["sigma"]) if k >= 0 else 0
        ans = k >= 0 and (inl >= 8 or agree >= 2)
        if not ans and rule == "STD6_then_15":
            cand = replay.pooled(photo, ids_all); k = replay.choose(cand); n = 15
            x, y, inl = (cand["x"][k], cand["y"][k], cand["inliers"][k]) if k >= 0 else (0, 0, 0)
            ans = k >= 0 and inl >= 11
    err = replay.error_sigma(photo, x, y) if ans else np.nan
    return dict(answered=bool(ans), err=err, points=replay.points(err) if ans else 0, n=n)
rows = []
for kind in ("clean", "height20"):
    d = replay.load_set(T7 / "rich_dump" / ("official60_scale15_" + kind))
    for rule, bar in [("STD6", 8), ("ALL15", 8), ("ALL15", 10), ("ALL15", 11), ("ALL15", 12), ("STD6_then_15", 11)]:
        res = [decide(p, rule, bar) for p in d.values()]
        rows.append(dict(kind=kind, rule=rule, bar=bar, score=round(replay.score([r["points"] for r in res]), 1), wrong=int(sum(r["answered"] and r["err"] > 8 for r in res)),
                         answered=round(np.mean([r["answered"] for r in res]), 3), queries=round(np.mean([r["n"] for r in res]), 1)))
R = pd.DataFrame(rows); R.to_csv(T7 / "idea1_scale_coverage.csv", index=False); print(R.to_string())
